# 02 · Context Length ve DataLoader

**Belgedeki bölüm:** Context Length, DataLoader

- Context length: modelin tek seferde görebildiği en uzun token dizisi. Sistem talimatı, kullanıcı girdisi ve modelin cevabı aynı pencereye sığmalı.
- Dil modeli, veri birer token kaydırılarak (girdi → hedef) bir sonraki token'ı tahmin etmek üzere eğitilir.
- DataLoader: veriyi batch'lere böler, karıştırır (shuffle) ve context length'e göre kırpar ya da `<pad>` ile doldurur.

In [ ]:
# the capital of france is paris

prompt = "the capital of the united states is not "

next_token = " "

output = "the capital of france is paris, "

In [ ]:
data = "the capital <pad> <pad> <pad> <pad> "
hedef = "capital of <pad> <pad> <pad> <pad> "

In [ ]:
context_length = 12 # gpt-4o 128k, gemma-3-1b 32K

In [ ]:
from usta_tokenizer import UstaTokenizer

tokenizer = UstaTokenizer("tokenizer.json")

ids = tokenizer.encode(prompt)

len(ids), ids

### Gerçek bir modelin context length'i
Modelin yapılandırma dosyası (config) en fazla kaç token görebildiğini söyler (küçük bir dosya indirir).

In [ ]:
from transformers import AutoConfig

# Gemma izinli (gated) olduğu için burada girişsiz indirilebilen Qwen3 kullanıldı
config = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B")
config.max_position_embeddings

In [ ]:
girdi = "the letter capital of the united states is not"
cikti = "letter capital of the united states is not london"

DataLoader Oluşturmadan Önce ilk Olarak Metnimizi Tokenlara ayırıyoruz

In [ ]:
with open("text.txt", "r") as f:
  text = f.read()

text

In [ ]:
token_ids = tokenizer.encode(text)
# save ids

ids_text = ""

for token_id in token_ids:
  ids_text += f"{token_id} "

with open("token_ids.txt", "w") as f:
  f.write(ids_text)

Text'i datasete çevirdikten sonra DataLoader Oluşturma

In [ ]:
import torch
from torch.utils.data import DataLoader
from text_dataset import TextDataset

stride = 12

def create_data_loader(token_ids: list, context_length: int, stride: int,
                       batch_size: int, shuffle: bool = True, device: str = "cpu"):
  dataset = TextDataset(token_ids, context_length, stride)
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      generator=torch.Generator(device=device)
    )
  
  return dataloader

In [ ]:
train_data_loader = create_data_loader(token_ids, context_length, stride, 1, False)

len(train_data_loader)

In [ ]:
i = 0
for batch in enumerate(train_data_loader):
  print(batch)
  i += 1
  if i > 2:
    break

In [ ]:
data_iter = iter(train_data_loader)

In [ ]:
next(data_iter)